# QF5210: four premia, four market structures

Runs every sleeve and the Monte Carlo forward test, in slide order. The logic lives in the
`book` package; this notebook only calls it and shows the results.

| Section | What | Slide | Module |
|---|---|---|---|
| 1 | E1 · PCA mean reversion (SGX + US) | 4 | `book/sleeves/e1_pca.py` |
| 2 | Wheel · BTC options to Jan 2025, then 70/30 HYPE/BTC | 5 | `book/sleeves/wheel.py` |
| 3 | Short IDR · long USD/IDR forwards with a dollar filter | 6 | `book/sleeves/fx_idr.py` |
| 4 | Dip · Nasdaq-100 dip buying | 7 | `book/sleeves/dip.py` |
| 5 | Monte Carlo forward test | A10 | `book/montecarlo.py` |

Frozen parameters are in `book/config.py`. Inputs go in `data/<sleeve>/`, results land in `outputs/<sleeve>/`.
The book is E1 / dip / short IDR / wheel at 20 / 15 / 15 / 50, rebalanced quarterly.
`scripts/fetch_data.py` downloads the inputs, and `scripts/book_report.py` computes the book-level numbers in the deck.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
from IPython.display import Image, display

from book import config, montecarlo as mc
from book.sleeves import dip, e1_pca, fx_idr, wheel

## 1. E1 · PCA statistical arbitrage (slide 4)
Top 3 PCA factors on a 60-day window; trade each name's OU s-score of the factor residual.
Open at |s| > 1.5, close at |s| < 0.5, stop at |s| > 3.5, half-life ≤ 30 days, 10 bps per weight change.

In [ ]:
e1 = e1_pca.run()

## 2. Wheel · BTC, then HYPE (slide 5)
Weekly 25-delta options. BTC priced off Deribit's DVOL to 3 Jan 2025, then 70% HYPE (priced off 30-day realised vol, no DVOL) and 30% BTC, rebalanced quarterly.
Each coin has its own 200-day rule: at a Friday close below its average, the BTC leg runs the next week at half size and the HYPE leg sits in T-bills.
The chart shows the adopted wheel against buy-and-hold of the same coins.


In [ ]:
wh = wheel.run()
eq = pd.read_csv(config.OUTPUTS / "wheel" / "returns_wheel_always.csv", index_col=0, parse_dates=True)["equity"]
bh = pd.read_csv(config.OUTPUTS / "wheel" / "returns_bh.csv", index_col=0, parse_dates=True)["equity"]
pd.DataFrame({"wheel (adopted)": eq, "buy and hold": bh}).plot(logy=True, figsize=(9, 4));

## 3. Short IDR · long USD/IDR forwards (slide 6)
Short the rupiah at 1x through rolling 1-month forwards, held only while the dollar index's 21-day return is positive.
Carry of 3% a year while on, 3 bps per monthly roll, 5 bps per switch.


In [ ]:
fx = fx_idr.run()
fx["equity"].plot(figsize=(9, 3), title="Short IDR sleeve, after carry and costs");

## 4. Dip · Nasdaq-100 dip buying (slide 7)

In [ ]:
dip_results = dip.run()
pd.DataFrame(dip_results).set_index("label")[
    ["cagr_pct", "sharpe", "max_dd_pct", "n_trades", "win_rate_pct", "avg_holding_days", "avg_capital_usage_pct"]]

## 5. Monte Carlo forward test (Appendix A10)

How the wheel could do over the next 4 weeks (the assessed window) and 13 weeks, starting from today's
live position. Each scenario has 2,000 paths:

- **Base:** stationary block bootstrap of daily HYPE moves and vol changes. Moves are scaled by the vol proxy
  (filtered historical simulation), so realised vol stays tied to the vol the options are priced at.
- **Noise:** each path redraws option pricing (0.83 to 1.0 × the vol proxy) and fill quality (85 to 95% of model price).
- **Shock:** the worst 7 days in the HYPE sample is replayed once at a random point in the horizon.

The calibration check asks whether the method would have worked in the past. At earlier dates it
forecasts with data up to that date only, then scores the actual outcome.

In [ ]:
live_hist, live_state = wheel.run_wheel(wh["market"], use_regime=False)
last_friday = live_hist.index[-1]
print(f"Live state {last_friday.date()}: cash ${live_state.cash:,.0f}, HYPE {live_state.qty:.3f}, open option {live_state.opt}")

tables = []
for sc in mc.SCENARIOS:
    paths, _, info = mc.wheel_forward(wh["market"], live_state, last_friday, scenario=sc)
    tables.append(mc.summarise(paths, config.MC_HORIZONS_WEEKS).assign(scenario=sc))
pd.concat(tables).set_index(["scenario", "weeks"])

In [ ]:
# walk-forward calibration (about 2 minutes)
cal = mc.wheel_calibration(wh["market"])
mc.calibration_summary(cal)

The full pipeline that writes `outputs/montecarlo/`, and renders the slide:
```
python scripts/run_montecarlo.py --calibration
python scripts/build_mc_slide.py
```

In [ ]:
slide = config.OUTPUTS / "montecarlo" / "slide_A10.png"
if slide.exists():
    display(Image(str(slide)))